# Módulo 1 · Etapa 2 — Análisis exploratorio (EDA)
**TFM · HAM10000** · Responsable: **Adrián Arias** · MLflow: experimento `m1-datos`, run `m1-eda`, tag `fase=eda`

**Objetivo:** que cada gráfico responda una pregunta y deje una conclusión escrita que se use después (capítulo 4 o Módulo 3).

| § | Pregunta | Para qué sirve |
|---|---|---|
| 2 | ¿Cuánto desbalance hay, en imágenes y en lesiones? | Etapa 5 (pesos por clase) |
| 3 | ¿Qué clases tienen más fotos repetidas de una misma lesión? | Justifica el split por lesión (Etapa 4) |
| 4 | ¿Qué etiquetas son más fiables (`dx_type`)? | Limitaciones |
| 5 | ¿Cómo se distribuye la edad por clase? | Descripción del dataset |
| 6 | ¿Y el sexo? | Descripción del dataset |
| 7 | ¿Dónde están las lesiones (localización)? | Descripción del dataset |
| 8 | ¿Cómo se ven las clases? | Figura para la memoria |
| 9 | ¿Alguna clase es más oscura o de otro tono? | Posible atajo (Módulo 3, constancia de color) |

**Alcance:** se analiza el **dataset completo**. Ninguna conclusión de aquí se usa para ajustar el modelo; lo que sí se use para decidir (pesos de clase) se calculará solo sobre train en la Etapa 5.

**Figuras:** se guardan en `TFM/figures/m1/etapa2/` a 300 ppp y **sin título dentro de la imagen**: según la plantilla, el número, el título y la fuente ("Elaboración propia a partir de HAM10000") se escriben en Word.

Entorno: **CPU**. Secretos: `DAGSHUB_USER`, `DAGSHUB_TOKEN`.

## 1. Preparación de la sesión

In [ ]:
AUTOR = "adrian"
OWNER, REPO = "aReinoso007", "tfm-melanoma-cnn-gradcam"
DATASET_SHA256_ESPERADO = "06aee5e603d313ba69de38263053c792a1dff030eb9c44a96ceb477ceefb99a5"   # huella registrada en la Etapa 0

In [ ]:
import os, time, shutil, zipfile
from pathlib import Path
from google.colab import drive, userdata
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/TFM")

REPO_DIR = f"/content/{REPO}"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/{OWNER}/{REPO}.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!pip install -q -r requirements.txt

In [ ]:
from src.tracking import load_config, file_sha256
cfg = load_config("configs/base.yaml")
LOCAL = Path(cfg["paths"]["local_data_dir"])
if not LOCAL.exists():
    t0 = time.time()
    shutil.copy(cfg["paths"]["dataset_zip"], "/content/HAM10000.zip")
    with zipfile.ZipFile("/content/HAM10000.zip") as z:
        z.extractall(LOCAL)
    os.remove("/content/HAM10000.zip")
    print(f"Descomprimido en {LOCAL} ({time.time()-t0:.0f} s)")

IMAGES = Path(cfg["paths"]["images_dir"])
META = Path(cfg["paths"]["metadata_csv"])
sha = file_sha256(META)
assert sha == DATASET_SHA256_ESPERADO, "[!] El CSV de metadatos no es el mismo que en la Etapa 0. Para y avísame."
print("[+] Mismo dataset que en la Etapa 0")

FIG = DRIVE_ROOT / "figures" / "m1" / "etapa2"
TAB = DRIVE_ROOT / "results" / "m1" / "etapa2"
FIG.mkdir(parents=True, exist_ok=True); TAB.mkdir(parents=True, exist_ok=True)

### Estilo común de las figuras
Paleta categórica fija y validada para daltonismo (los colores siguen siempre a la misma categoría), una sola tonalidad para magnitudes, rejilla discreta y números directamente sobre las barras.

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

CLASES = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]
NOMBRE = {"akiec": "Queratosis actínica / Bowen", "bcc": "Carcinoma basocelular",
          "bkl": "Lesión queratósica benigna", "df": "Dermatofibroma", "mel": "Melanoma",
          "nv": "Nevus melanocítico", "vasc": "Lesión vascular"}
AZUL = "#2a78d6"                                     # serie única / magnitud
CAT = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]   # categorías, en este orden fijo
GRIS = "#b9b8b0"                                     # "unknown"
TINTA, TINTA2 = "#1f1f1d", "#6b6a64"

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "font.size": 10, "axes.edgecolor": "#c9c8c1",
    "axes.labelcolor": TINTA2, "xtick.color": TINTA2, "ytick.color": TINTA2,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
    "grid.color": "#e8e7e2", "grid.linewidth": 0.8, "axes.axisbelow": True,
    "legend.frameon": False,
})

def es(x, dec=1):
    """Número con formato español: miles con punto y decimales con coma."""
    return f"{x:,.{dec}f}".replace(",", "X").replace(".", ",").replace("X", ".")

def guardar(fig, nombre):
    fig.savefig(FIG / f"{nombre}.png", bbox_inches="tight", facecolor="white")
    plt.show()

def barras_100(tabla, colores, ax, etiqueta_min=6):
    """Barras horizontales apiladas al 100 % (una por clase), con huecos blancos entre segmentos."""
    izq = np.zeros(len(tabla))
    for col, color in zip(tabla.columns, colores):
        v = tabla[col].values
        ax.barh(tabla.index, v, left=izq, color=color, edgecolor="white", linewidth=2,
                height=0.62, label=col)
        for y, (x0, w) in enumerate(zip(izq, v)):
            if w >= etiqueta_min:
                ax.text(x0 + w / 2, y, f"{es(w, 0)} %", ha="center", va="center",
                        color="white", fontsize=8, fontweight="bold")
        izq += v
    ax.set_xlim(0, 100); ax.xaxis.set_major_formatter(PercentFormatter())
    ax.invert_yaxis(); ax.grid(axis="y", visible=False)
    ax.legend(ncol=len(tabla.columns), loc="lower left", bbox_to_anchor=(0, 1.0), fontsize=9)

meta = pd.read_csv(META)
print(len(meta), "imágenes cargadas")

## 2. Distribución de clases: imágenes y lesiones

In [ ]:
dist = pd.DataFrame({"imagenes": meta.dx.value_counts(),
                     "lesiones": meta.drop_duplicates("lesion_id").dx.value_counts()}).reindex(CLASES)
dist["pct_imagenes"] = (100 * dist.imagenes / dist.imagenes.sum()).round(2)
dist["pct_lesiones"] = (100 * dist.lesiones / dist.lesiones.sum()).round(2)
dist.insert(0, "nombre", dist.index.map(NOMBRE))
dist.to_csv(TAB / "distribucion_clases.csv")
display(dist)

orden = dist.sort_values("imagenes").index          # de menor a mayor: la mayor queda arriba
fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.barh(orden, dist.loc[orden, "imagenes"], color=AZUL, height=0.62)
for y, c in enumerate(orden):
    r = dist.loc[c]
    ax.text(r.imagenes + dist.imagenes.max() * 0.012, y, f"{es(r.imagenes, 0)} img ({es(r.pct_imagenes)} %) · {es(r.lesiones, 0)} lesiones",
            va="center", fontsize=8.5, color=TINTA)
ax.set_xlim(0, dist.imagenes.max() * 1.55); ax.grid(axis="y", visible=False)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: es(v, 0)))
ax.set_xlabel("Número de imágenes")
guardar(fig, "fig_distribucion_clases")

ratio = dist.imagenes.max() / dist.imagenes.min()
print(f"Razón de desbalance (clase mayor / menor): {ratio:.1f}")
print(f"Clases con menos del 5 % de las imágenes: {list(dist.index[dist.pct_imagenes < 5])}")

## 3. Imágenes por lesión, por clase

In [ ]:
por_lesion = meta.groupby(["lesion_id", "dx"]).size().rename("n").reset_index()
rep = por_lesion.groupby("dx").agg(lesiones=("n", "size"), media_img=("n", "mean"),
                                   max_img=("n", "max"),
                                   pct_varias=("n", lambda s: 100 * (s > 1).mean())).reindex(CLASES)
# % de IMÁGENES que pertenecen a una lesión con más de una foto (lo que generaría fuga en un split por imagen)
img_multi = meta.merge(por_lesion[["lesion_id", "n"]], on="lesion_id")
rep["pct_img_en_lesion_repetida"] = img_multi.assign(m=img_multi.n > 1).groupby("dx").m.mean().reindex(CLASES) * 100
rep = rep.round(2); rep.to_csv(TAB / "imagenes_por_lesion.csv"); display(rep)

orden = rep.sort_values("pct_img_en_lesion_repetida").index
fig, ax = plt.subplots(figsize=(7.5, 3.4))
ax.barh(orden, rep.loc[orden, "pct_img_en_lesion_repetida"], color=AZUL, height=0.62)
for y, c in enumerate(orden):
    ax.text(rep.loc[c, "pct_img_en_lesion_repetida"] + 1, y,
            f"{es(rep.loc[c, 'pct_img_en_lesion_repetida'], 0)} % · media {es(rep.loc[c, 'media_img'], 2)} img/lesión",
            va="center", fontsize=8.5, color=TINTA)
ax.set_xlim(0, 100); ax.xaxis.set_major_formatter(PercentFormatter()); ax.grid(axis="y", visible=False)
ax.set_xlabel("Imágenes que pertenecen a una lesión fotografiada más de una vez")
guardar(fig, "fig_imagenes_por_lesion")
print("Global:", f"{100 * (img_multi.n > 1).mean():.1f} % de las imágenes están en lesiones con varias fotos")

## 4. Método de confirmación del diagnóstico (`dx_type`) por clase
`histo` = histopatología (biopsia, el más fiable) · `follow_up` = seguimiento sin cambios · `consensus` = consenso de expertos · `confocal` = microscopía confocal.

In [ ]:
ORDEN_DX = ["histo", "consensus", "confocal", "follow_up"]
dxt = pd.crosstab(meta.dx, meta.dx_type, normalize="index").reindex(index=CLASES, columns=ORDEN_DX).fillna(0) * 100
dxt_n = pd.crosstab(meta.dx, meta.dx_type).reindex(index=CLASES, columns=ORDEN_DX).fillna(0).astype(int)
dxt_n.to_csv(TAB / "dx_type_por_clase.csv"); display(dxt.round(1))

fig, ax = plt.subplots(figsize=(7.5, 3.6))
barras_100(dxt, CAT, ax)
ax.set_xlabel("Porcentaje de imágenes de la clase")
guardar(fig, "fig_dx_type_por_clase")
print("Clases 100 % confirmadas por histopatología:", list(dxt.index[dxt.histo >= 99.99]))

## 5. Edad por clase
Las edades vienen en intervalos de 5 años. Se revisan aparte las edades de 0 años.

In [ ]:
edad = meta.dropna(subset=["age"])
res_edad = edad.groupby("dx").age.describe()[["count", "mean", "25%", "50%", "75%", "min", "max"]].reindex(CLASES).round(1)
res_edad.to_csv(TAB / "edad_por_clase.csv"); display(res_edad)

orden = res_edad.sort_values("50%").index
fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.boxplot([edad.loc[edad.dx == c, "age"] for c in orden], vert=False, tick_labels=list(orden),
           widths=0.55, patch_artist=True, showfliers=True,
           boxprops=dict(facecolor="#cde2fb", edgecolor=AZUL, linewidth=1.2),
           medianprops=dict(color=AZUL, linewidth=2), whiskerprops=dict(color=AZUL),
           capprops=dict(color=AZUL), flierprops=dict(marker="o", markersize=3, markerfacecolor=AZUL,
                                                     markeredgecolor="none", alpha=0.4))
ax.set_xlabel("Edad (años)"); ax.grid(axis="y", visible=False)
guardar(fig, "fig_edad_por_clase")

cero = meta[meta.age == 0]
print(f"Imágenes con edad 0: {len(cero)} · por clase: {cero.dx.value_counts().to_dict()}")

## 6. Sexo por clase

In [ ]:
sx = pd.crosstab(meta.dx, meta.sex, normalize="index").reindex(index=CLASES, columns=["male", "female", "unknown"]).fillna(0) * 100
pd.crosstab(meta.dx, meta.sex).reindex(CLASES).to_csv(TAB / "sexo_por_clase.csv"); display(sx.round(1))

fig, ax = plt.subplots(figsize=(7.5, 3.6))
barras_100(sx.rename(columns={"male": "hombre", "female": "mujer", "unknown": "desconocido"}),
           [CAT[0], CAT[1], GRIS], ax)
ax.axvline(50, color=TINTA2, linewidth=0.8, linestyle=":")
ax.set_xlabel("Porcentaje de imágenes de la clase")
guardar(fig, "fig_sexo_por_clase")

## 7. Localización anatómica por clase
Cada fila suma 100 %: qué parte del cuerpo concentra las lesiones de cada clase.

In [ ]:
loc_n = pd.crosstab(meta.localization, meta.dx).reindex(columns=CLASES)
loc_n = loc_n.loc[loc_n.sum(axis=1).sort_values(ascending=False).index]
loc_n.to_csv(TAB / "localizacion_por_clase.csv")
loc_pct = (100 * loc_n / loc_n.sum(axis=0)).T          # filas = clase, columnas = localización

fig, ax = plt.subplots(figsize=(10, 3.9))
im = ax.imshow(loc_pct.values, cmap="Blues", vmin=0, vmax=max(40, loc_pct.values.max()), aspect="auto")
ax.set_xticks(range(loc_pct.shape[1]), loc_pct.columns, rotation=40, ha="right")
ax.set_yticks(range(len(CLASES)), CLASES); ax.grid(False)
for i in range(loc_pct.shape[0]):
    for j in range(loc_pct.shape[1]):
        v = loc_pct.values[i, j]
        if v >= 1:
            ax.text(j, i, es(v, 0), ha="center", va="center", fontsize=7.5,
                    color="white" if v > 25 else TINTA)
cb = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.01); cb.ax.yaxis.set_major_formatter(PercentFormatter())
cb.outline.set_visible(False)
guardar(fig, "fig_localizacion_por_clase")
print("Localización más frecuente por clase:", loc_pct.idxmax(axis=1).to_dict())

## 8. Ejemplos por clase
6 imágenes al azar por clase (semilla fija), para ver la variedad dentro de cada una.

In [ ]:
from PIL import Image
K, SEED = 6, 42
fig, axes = plt.subplots(len(CLASES), K, figsize=(1.9 * K, 1.5 * len(CLASES)))
for r, c in enumerate(CLASES):
    ids = meta[meta.dx == c].sample(K, random_state=SEED).image_id
    for j, iid in enumerate(ids):
        ax = axes[r, j]; ax.imshow(Image.open(IMAGES / f"{iid}.jpg")); ax.set_xticks([]); ax.set_yticks([])
        for s in ax.spines.values(): s.set_visible(False)
    axes[r, 0].set_ylabel(c, rotation=0, ha="right", va="center", fontsize=10, color=TINTA, labelpad=8)
fig.subplots_adjust(wspace=0.03, hspace=0.06)
guardar(fig, "fig_ejemplos_por_clase")

## 9. Brillo y color medio por clase (posible atajo)
Se mide sobre **toda la imagen** en las 10.015 (≈ 1–2 min). Si una clase es sistemáticamente más oscura o de otro tono, el modelo podría aprender ese rasgo en lugar de la lesión: es una pista para el Módulo 3 y para decidir si se prueba la constancia de color en la ablación.

- **Brillo** = luminancia media (0 negro · 255 blanco).
- **R, G, B** = media de cada canal. **Esquinas** = brillo de las cuatro esquinas (si es muy bajo, hay borde negro del dermatoscopio; se estudia a fondo en la Etapa 3).

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def color_medio(iid):
    with Image.open(IMAGES / f"{iid}.jpg") as im:
        im.draft("RGB", (150, 113))                 # decodifica a menor tamaño: mucho más rápido
        a = np.asarray(im.convert("RGB"), dtype=np.float32)
    lum = 0.299 * a[..., 0] + 0.587 * a[..., 1] + 0.114 * a[..., 2]
    h, w = lum.shape; k = max(2, h // 10)
    esquinas = np.r_[lum[:k, :k].ravel(), lum[:k, -k:].ravel(), lum[-k:, :k].ravel(), lum[-k:, -k:].ravel()]
    return dict(image_id=iid, R=a[..., 0].mean(), G=a[..., 1].mean(), B=a[..., 2].mean(),
                brillo=lum.mean(), esquinas=esquinas.mean())

t0 = time.time()
with ThreadPoolExecutor(8) as ex:
    col = pd.DataFrame(list(ex.map(color_medio, meta.image_id))).astype({v: "float64" for v in ["R", "G", "B", "brillo", "esquinas"]})
col = col.merge(meta[["image_id", "dx"]], on="image_id")
col.to_csv(TAB / "color_por_imagen.csv", index=False)
print(f"{len(col)} imágenes en {time.time()-t0:.0f} s")

res_col = col.groupby("dx")[["brillo", "R", "G", "B", "esquinas"]].median().reindex(CLASES).round(1)
res_col["pct_esquinas_oscuras"] = col.assign(o=col.esquinas < 40).groupby("dx").o.mean().reindex(CLASES).mul(100).round(1)
res_col.to_csv(TAB / "color_por_clase.csv"); display(res_col)

In [ ]:
orden = res_col.sort_values("brillo").index
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for ax, var, xl in [(axes[0], "brillo", "Brillo medio de la imagen (0–255)"),
                    (axes[1], "esquinas", "Brillo de las esquinas (0–255)")]:
    ax.boxplot([col.loc[col.dx == c, var] for c in orden], vert=False, tick_labels=list(orden), widths=0.55,
               patch_artist=True, showfliers=False,
               boxprops=dict(facecolor="#cde2fb", edgecolor=AZUL, linewidth=1.2),
               medianprops=dict(color=AZUL, linewidth=2), whiskerprops=dict(color=AZUL), capprops=dict(color=AZUL))
    ax.set_xlabel(xl); ax.set_xlim(0, 255); ax.grid(axis="y", visible=False)
fig.tight_layout()
guardar(fig, "fig_brillo_por_clase")

# Tamaño del efecto: diferencia de medianas respecto a nv, en desviaciones típicas de nv
ref = col[col.dx == "nv"]
for var in ["brillo", "esquinas"]:
    d = {c: round(float((col.loc[col.dx == c, var].median() - ref[var].median()) / ref[var].std()), 2) for c in CLASES}
    print(f"{var}: diferencia de mediana frente a nv (en DE de nv): {d}")
print("Referencia: |diferencia| < 0,2 despreciable · ~0,5 moderada · > 0,8 grande")

## 10. Registro en MLflow

In [ ]:
import mlflow
from src.tracking import setup_mlflow, standard_tags
os.environ["MLFLOW_TRACKING_USERNAME"] = userdata.get("DAGSHUB_USER")
os.environ["MLFLOW_TRACKING_PASSWORD"] = userdata.get("DAGSHUB_TOKEN")
setup_mlflow(cfg)
mlflow.set_experiment(cfg["mlflow"]["experiments"]["m1"])     # m1-datos

tags = standard_tags(cfg, modulo="m1", fase="eda", autor=AUTOR, dataset_sha256=sha)
with mlflow.start_run(run_name="m1-eda", tags=tags) as run:
    mlflow.log_params({"alcance": "dataset_completo", "muestras_por_clase_figura": K, "semilla_muestras": SEED})
    mlflow.log_metrics({
        "ratio_desbalance": float(ratio),
        "pct_nv": float(dist.loc["nv", "pct_imagenes"]),
        "pct_img_en_lesion_repetida": float(100 * (img_multi.n > 1).mean()),
        "pct_histo": float(100 * (meta.dx_type == "histo").mean()),
        "n_edad_cero": int(len(cero)),
    })
    mlflow.log_artifacts(str(FIG), artifact_path="figuras")
    mlflow.log_artifacts(str(TAB), artifact_path="tablas")
print("[+] Run registrado:", run.info.run_id)
print("Figuras:", sorted(p.name for p in FIG.iterdir()))